# <center>Optuna: Hyperparameter Optimization for Kaggle</center>

<center>

![Python](https://img.shields.io/badge/Python-3.10-blue?logo=python&logoColor=white)
![Optuna](https://img.shields.io/badge/Optuna-3.x-blueviolet)
![XGBoost](https://img.shields.io/badge/XGBoost-2.0-green)
![LightGBM](https://img.shields.io/badge/LightGBM-4.1-purple)
![scikit-learn](https://img.shields.io/badge/scikit--learn-1.3-orange?logo=scikit-learn)
![License](https://img.shields.io/badge/License-MIT-red)

</center>

---

**Author:** Lorenzo Scaturchio  
**Last Updated:** February 2026  
**Kernel Version:** 1.0

> *"Hyperparameter tuning is where competitions are won or lost. Optuna is the sharpest tool in the box."*  
> -- Kaggle Grandmaster Handbook

---

## TL;DR

This notebook is a **complete, practical guide** to hyperparameter optimization with Optuna -- the framework used by top Kaggle competitors worldwide.

| Method | Strategy | Sample Efficiency | Best For |
|--------|----------|------------------|----------|
| **Grid Search** | Exhaustive enumeration | Very Low | Small, discrete spaces |
| **Random Search** | Uniform random sampling | Low | Baseline exploration |
| **Bayesian (Optuna/TPE)** | Probabilistic surrogate model | **High** | Any continuous/mixed space |
| **CMA-ES** | Evolution strategy | High | High-dim continuous spaces |

**Key takeaway:** Optuna's TPE sampler typically finds results in 50 trials that Random Search needs 500+ trials to match.

If you find this notebook useful, please consider giving it an **upvote** -- it helps others discover it!

## Table of Contents

1. [Why Hyperparameter Optimization Matters](#1)
2. [Setup & Data](#2)
3. [Optuna Basics: Your First Study](#3)
4. [TPE Sampler: How It Works](#4)
5. [CMA-ES Sampler: Continuous Spaces](#5)
6. [Pruning: Kill Unpromising Trials Early](#6)
7. [Visualizing Studies](#7)
8. [XGBoost + Optuna: Production Pipeline](#8)
9. [LightGBM + Optuna: Production Pipeline](#9)
10. [Sklearn Integration: OptunaSearchCV](#10)
11. [Multi-Objective Optimization](#11)
12. [Hyperparameter Importance](#12)
13. [Production Tips for Kaggle](#13)

---

<a id="1"></a>
## 1. Why Hyperparameter Optimization Matters

### The Cost of Bad Hyperparameters

Machine learning models are extraordinarily sensitive to hyperparameter choices. Consider a gradient-boosted tree:

- **Learning rate**: span of 4 orders of magnitude (0.001 to 1.0)
- **Max depth**: integer from 2 to 12
- **Subsample**: continuous from 0.5 to 1.0
- **colsample_bytree**: continuous from 0.5 to 1.0
- **reg_alpha, reg_lambda**: log-scale over 6 orders of magnitude each

A naive grid search over just 5 values per parameter = **5^6 = 15,625 evaluations**. At 30 seconds each, that is 130 hours. Kaggle kernels give you 12.

### Comparison of Strategies

| Strategy | Trials to find top-5% params | Wall time (est.) | Handles dependencies? |
|----------|-------------------------------|------------------|-----------------------|
| Grid Search | 10,000+ | Days | No |
| Random Search | 200-500 | Hours | Partially |
| **Bayesian (Optuna)** | **50-100** | **Minutes** | **Yes** |

### Why Bayesian Optimization Wins

Bayesian optimization maintains a **probabilistic surrogate model** of the objective function. After each trial:

1. The model is updated with the new (params, score) observation
2. An **acquisition function** (EI -- Expected Improvement) selects the next point to evaluate
3. Points in promising regions are sampled more frequently

This is fundamentally different from random search: instead of sampling blindly, the algorithm *learns* which regions of the search space are worth exploring.

**Optuna's TPE (Tree-structured Parzen Estimator)** extends this to handle:
- Mixed integer/float/categorical parameters
- Conditional search spaces (e.g., "only tune dropout if using neural net")
- Parallel execution across multiple cores or machines

---

<a id="2"></a>
## 2. Setup & Data

Install Optuna and its integration package, then load the breast cancer dataset as our benchmark throughout this guide.

In [ ]:
!pip install optuna optuna-integration -q

import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import time
import warnings

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    cross_val_score, StratifiedKFold, train_test_split
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Reproducibility
np.random.seed(42)

# Styling
plt.style.use("seaborn-v0_8-whitegrid")
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

# Dataset: breast cancer (binary classification, 569 samples, 30 features)
X, y = load_breast_cancer(return_X_y=True)
feature_names = load_breast_cancer().feature_names
print(f"Dataset shape : {X.shape}")
print(f"Positive class: {y.mean():.2%}")
print(f"\nFeatures: {list(feature_names[:5])} ...")

<a id="3"></a>
## 3. Optuna Basics: Your First Study

### Core Vocabulary

| Term | Definition |
|------|------------|
| **Trial** | A single call to the objective function with one set of hyperparameters |
| **Study** | A collection of trials; persists the optimization history |
| **Objective** | The Python function you want to maximize or minimize |
| **Sampler** | The algorithm that proposes new parameter values (TPE, CMA-ES, Random) |
| **Pruner** | An early-stopping rule that kills unpromising trials mid-run |

### Suggest API

Inside the objective, use `trial.suggest_*` to declare hyperparameters:

```python
trial.suggest_int("n_estimators", 50, 300)          # integer
trial.suggest_float("learning_rate", 1e-4, 0.3, log=True)  # float, log scale
trial.suggest_float("dropout", 0.1, 0.5)            # float, linear scale
trial.suggest_categorical("criterion", ["gini", "entropy"])  # categorical
```

The `log=True` flag is **critical** for parameters like learning rates that span multiple orders of magnitude -- it ensures the sampler allocates equal probability mass to each order of magnitude rather than biasing toward large values.

In [ ]:
def objective(trial):
    """Simple RandomForest objective for breast cancer classification."""
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 3, 10)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])

    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42,
        n_jobs=-1,
    )
    score = cross_val_score(
        clf, X, y, cv=5, scoring="roc_auc", n_jobs=-1
    ).mean()
    return score


# Create and run a study
study = optuna.create_study(direction="maximize", study_name="rf_baseline")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nBest trial : #{study.best_trial.number}")
print(f"Best ROC-AUC: {study.best_value:.4f}")
print(f"Best params :")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Inspect all trials as a DataFrame
trials_df = study.trials_dataframe()
print(f"Columns: {list(trials_df.columns)}\n")

# Sort by value and display top 5
top5 = trials_df.sort_values("value", ascending=False).head(5)
display_cols = ["number", "value", "params_n_estimators", "params_max_depth",
                "params_min_samples_split", "params_min_samples_leaf", "params_max_features"]
print("Top 5 trials:")
print(top5[display_cols].to_string(index=False))

<a id="4"></a>
## 4. TPE Sampler: How It Works

### Tree-structured Parzen Estimator (TPE)

TPE is Optuna's default sampler and the reason it outperforms random search. The algorithm works as follows:

1. **Split trials** into "good" (top gamma%) and "bad" (bottom 1-gamma%) based on objective value
2. **Fit two density estimators**: `l(x)` over good trials, `g(x)` over bad trials
3. **Sample new candidate** by maximizing `l(x) / g(x)` (Expected Improvement proxy)
4. **Evaluate** the candidate and update the model

This means TPE *learns the distribution of good hyperparameters* and concentrates sampling there.

### TPE vs Random Search: Head-to-Head

Let's run both with the same budget and compare convergence speed.

In [ ]:
# Random baseline
random_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.RandomSampler(seed=42),
    study_name="rf_random",
)
random_study.optimize(objective, n_trials=60, show_progress_bar=False)

# TPE (default)
tpe_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="rf_tpe",
)
tpe_study.optimize(objective, n_trials=60, show_progress_bar=False)

# Plot convergence curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for study_obj, label, color in [
    (random_study, "Random Search", PALETTE[2]),
    (tpe_study, "TPE (Optuna)", PALETTE[0]),
]:
    values = [t.value for t in study_obj.trials]
    best_so_far = pd.Series(values).cummax()
    axes[0].plot(best_so_far.values, label=label, color=color, linewidth=2.5)
    axes[1].plot(values, label=label, color=color, alpha=0.7, linewidth=1.5)

axes[0].set_title("Best ROC-AUC Found So Far", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Trial Number")
axes[0].set_ylabel("Best ROC-AUC")
axes[0].legend()

axes[1].set_title("Per-Trial ROC-AUC (All Trials)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Trial Number")
axes[1].set_ylabel("ROC-AUC")
axes[1].legend()

plt.suptitle("Random Search vs TPE Convergence", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"Random Search best: {random_study.best_value:.4f} (trial #{random_study.best_trial.number})")
print(f"TPE best          : {tpe_study.best_value:.4f} (trial #{tpe_study.best_trial.number})")

<a id="5"></a>
## 5. CMA-ES Sampler: Continuous Spaces

### When to Use CMA-ES

CMA-ES (Covariance Matrix Adaptation Evolution Strategy) is an evolutionary algorithm optimized for **high-dimensional, continuous** search spaces. Use it when:

- Most hyperparameters are **floats** (not integers or categoricals)
- The space has **more than 5 dimensions**
- Parameters are **correlated** (e.g., learning_rate and n_estimators interact)

CMA-ES maintains a **multivariate Gaussian** over the search space and adapts both the mean and covariance matrix based on successful candidates, naturally learning parameter correlations.

### CMA-ES on a Float-Heavy XGBoost Space

In [ ]:
def objective_continuous(trial):
    """XGBoost with float-heavy search space -- ideal for CMA-ES."""
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    reg_alpha = trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True)
    reg_lambda = trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True)
    gamma = trial.suggest_float("gamma", 0.0, 5.0)
    min_child_weight = trial.suggest_float("min_child_weight", 1.0, 10.0)

    clf = xgb.XGBClassifier(
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        gamma=gamma,
        min_child_weight=min_child_weight,
        n_estimators=100,
        eval_metric="auc",
        random_state=42,
        verbosity=0,
    )
    score = cross_val_score(
        clf, X, y, cv=5, scoring="roc_auc", n_jobs=-1
    ).mean()
    return score


# TPE on float space (baseline)
tpe_cont_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
tpe_cont_study.optimize(objective_continuous, n_trials=60, show_progress_bar=False)

# CMA-ES on float space
cmaes_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.CmaEsSampler(seed=42),
)
cmaes_study.optimize(objective_continuous, n_trials=60, show_progress_bar=False)

# Convergence comparison
fig, ax = plt.subplots(figsize=(11, 5))
for study_obj, label, color in [
    (tpe_cont_study, "TPE", PALETTE[0]),
    (cmaes_study, "CMA-ES", PALETTE[1]),
]:
    values = [t.value for t in study_obj.trials]
    best_so_far = pd.Series(values).cummax()
    ax.plot(best_so_far.values, label=label, color=color, linewidth=2.5)

ax.set_title("TPE vs CMA-ES on Continuous XGBoost Space", fontsize=13, fontweight="bold")
ax.set_xlabel("Trial Number")
ax.set_ylabel("Best ROC-AUC")
ax.legend()
plt.tight_layout()
plt.show()

print(f"TPE best  : {tpe_cont_study.best_value:.4f}")
print(f"CMA-ES best: {cmaes_study.best_value:.4f}")
print(f"\nCMA-ES best params:")
for k, v in cmaes_study.best_params.items():
    print(f"  {k}: {v:.6g}")

<a id="6"></a>
## 6. Pruning: Kill Unpromising Trials Early

### What is Pruning?

Pruning is **early stopping at the study level**: if a trial's intermediate scores are consistently below the median of completed trials, kill it and move on. This can reduce total compute by 30-70%.

### MedianPruner

The most common pruner in Optuna:

```python
optuna.pruners.MedianPruner(
    n_startup_trials=5,   # run first N trials fully (cold start)
    n_warmup_steps=10,    # skip pruning for first N steps of each trial
    interval_steps=1,     # check pruning every N steps
)
```

### XGBoost Integration with Pruning Callback

Optuna's integration package provides `XGBoostPruningCallback` which reports intermediate scores after each boosting round and asks Optuna whether to continue.

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


def objective_with_pruning(trial):
    """XGBoost objective that supports Optuna pruning via callback."""
    params = {
        "n_estimators": 500,
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
        "eval_metric": "auc",
        "early_stopping_rounds": 30,
        "random_state": 42,
        "verbosity": 0,
    }

    try:
        from optuna.integration import XGBoostPruningCallback
        callbacks = [XGBoostPruningCallback(trial, "validation_0-auc")]
    except ImportError:
        callbacks = []

    clf = xgb.XGBClassifier(**params)
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
        callbacks=callbacks,
    )
    score = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
    return score


# Without pruning
no_prune_study = optuna.create_study(direction="maximize")
t0 = time.time()
no_prune_study.optimize(objective_with_pruning, n_trials=30, show_progress_bar=False)
no_prune_time = time.time() - t0

# With MedianPruner
pruning_study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=15,
    ),
)
t0 = time.time()
pruning_study.optimize(objective_with_pruning, n_trials=30, show_progress_bar=False)
prune_time = time.time() - t0

n_pruned = sum(
    1 for t in pruning_study.trials
    if t.state == optuna.trial.TrialState.PRUNED
)
print(f"=== Without Pruning ===")
print(f"  Best AUC  : {no_prune_study.best_value:.4f}")
print(f"  Wall time : {no_prune_time:.1f}s")
print(f"\n=== With MedianPruner ===")
print(f"  Best AUC  : {pruning_study.best_value:.4f}")
print(f"  Wall time : {prune_time:.1f}s")
print(f"  Pruned    : {n_pruned}/{len(pruning_study.trials)} trials ({n_pruned/len(pruning_study.trials):.0%})")
print(f"  Speedup   : {no_prune_time/max(prune_time,1):.1f}x")

<a id="7"></a>
## 7. Visualizing Studies

Optuna ships with a rich visualization module built on **Plotly**. These plots are interactive and render natively in Kaggle notebooks.

| Plot | Purpose |
|------|---------|
| `plot_optimization_history` | Track best value over trials |
| `plot_slice` | How each param affects the objective |
| `plot_contour` | 2D interaction between two params |
| `plot_parallel_coordinate` | All params vs objective in one view |
| `plot_param_importances` | fANOVA importance ranking |
| `plot_edf` | Empirical distribution of trial values |

In [ ]:
# We use the tpe_study from Section 3 for all visualizations

# 1. Optimization history
fig = optuna.visualization.plot_optimization_history(tpe_study)
fig.update_layout(title="Optimization History (TPE Study)", height=400)
fig.show()

# 2. Slice plot -- how each param affects objective
fig = optuna.visualization.plot_slice(tpe_study)
fig.update_layout(title="Slice Plot: Per-Parameter Effect on ROC-AUC", height=400)
fig.show()

# 3. Contour: interaction between n_estimators and max_depth
fig = optuna.visualization.plot_contour(
    tpe_study, params=["n_estimators", "max_depth"]
)
fig.update_layout(title="Contour: n_estimators vs max_depth", height=450)
fig.show()

# 4. Parallel coordinate -- overview of all trials
fig = optuna.visualization.plot_parallel_coordinate(tpe_study)
fig.update_layout(title="Parallel Coordinates: All Hyperparameters", height=450)
fig.show()

# 5. EDF -- how value is distributed across trials
fig = optuna.visualization.plot_edf(tpe_study)
fig.update_layout(title="Empirical Distribution of Trial Values", height=400)
fig.show()

In [ ]:
# Matplotlib fallback: convergence + trial value distribution
# (useful if Plotly is unavailable or for static export)

values = [t.value for t in tpe_study.trials]
best_so_far = pd.Series(values).cummax()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: convergence
axes[0].plot(best_so_far.values, color=PALETTE[0], linewidth=2.5, label="Best so far")
axes[0].scatter(
    range(len(values)), values,
    color=PALETTE[0], alpha=0.25, s=15, label="Individual trial"
)
axes[0].axhline(
    tpe_study.best_value, color="red", linestyle="--", linewidth=1.5,
    label=f"Best = {tpe_study.best_value:.4f}"
)
axes[0].set_title("Optimization History (Matplotlib)", fontweight="bold")
axes[0].set_xlabel("Trial Number")
axes[0].set_ylabel("ROC-AUC")
axes[0].legend()

# Right: distribution of trial values
axes[1].hist(values, bins=20, color=PALETTE[0], edgecolor="white", alpha=0.85)
axes[1].axvline(
    tpe_study.best_value, color="red", linestyle="--", linewidth=2,
    label=f"Best = {tpe_study.best_value:.4f}"
)
axes[1].set_title("Distribution of Trial Values", fontweight="bold")
axes[1].set_xlabel("ROC-AUC")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.suptitle("Study Visualization (Matplotlib)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

<a id="8"></a>
## 8. XGBoost + Optuna: Production Pipeline

Here is a full, competition-ready XGBoost objective covering all major hyperparameters. Key design decisions:

- **StratifiedKFold** ensures balanced class distribution in each fold
- **Early stopping** per fold prevents overfitting and speeds evaluation
- **Log-scale** for learning rate, alpha, lambda
- After optimization: **retrain on full dataset** with best params

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def xgb_objective(trial):
    """Production XGBoost objective for Kaggle competitions."""
    params = {
        # Boosting params
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        # Sampling
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.4, 1.0),
        # Regularization
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        # Fixed
        "eval_metric": "auc",
        "tree_method": "hist",
        "random_state": 42,
        "verbosity": 0,
        "early_stopping_rounds": 50,
    }

    oof_scores = []
    for fold_idx, (train_idx, val_idx) in enumerate(CV.split(X, y)):
        X_fold_tr, X_fold_val = X[train_idx], X[val_idx]
        y_fold_tr, y_fold_val = y[train_idx], y[val_idx]

        model = xgb.XGBClassifier(**params)
        model.fit(
            X_fold_tr, y_fold_tr,
            eval_set=[(X_fold_val, y_fold_val)],
            verbose=False,
        )
        preds = model.predict_proba(X_fold_val)[:, 1]
        oof_scores.append(roc_auc_score(y_fold_val, preds))

    return float(np.mean(oof_scores))


xgb_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
    study_name="xgb_production",
)
xgb_study.optimize(xgb_objective, n_trials=40, show_progress_bar=True)

print(f"\nXGBoost best CV ROC-AUC: {xgb_study.best_value:.4f}")
print(f"Best params:")
for k, v in xgb_study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Retrain best XGBoost on the full dataset
best_xgb_params = dict(xgb_study.best_params)
best_xgb_params.update({
    "eval_metric": "auc",
    "tree_method": "hist",
    "random_state": 42,
    "verbosity": 0,
})
# Remove early_stopping_rounds for full retraining (no val set needed)
best_xgb_params.pop("early_stopping_rounds", None)

best_xgb = xgb.XGBClassifier(**best_xgb_params)
best_xgb.fit(X, y)

# Feature importance plot
importances = best_xgb.feature_importances_
top_n = 15
top_idx = np.argsort(importances)[-top_n:][::-1]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    [feature_names[i] for i in top_idx][::-1],
    importances[top_idx][::-1],
    color=PALETTE[0], edgecolor="white"
)
ax.set_title(f"Top {top_n} Feature Importances (Tuned XGBoost)", fontweight="bold")
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.show()

print(f"Model trained on {X.shape[0]} samples with {X.shape[1]} features.")

<a id="9"></a>
## 9. LightGBM + Optuna: Production Pipeline

LightGBM has a different set of key hyperparameters compared to XGBoost:

| Parameter | Description | Typical Range |
|-----------|-------------|---------------|
| `num_leaves` | Max leaves per tree (complexity) | 20 -- 300 |
| `min_child_samples` | Min data per leaf (regularization) | 5 -- 100 |
| `feature_fraction` | Column subsampling ratio | 0.4 -- 1.0 |
| `bagging_fraction` | Row subsampling ratio | 0.4 -- 1.0 |
| `bagging_freq` | Bagging frequency | 1 -- 7 |
| `reg_alpha` | L1 regularization | 1e-8 -- 10 |
| `reg_lambda` | L2 regularization | 1e-8 -- 10 |

Note: `num_leaves` is more important than `max_depth` in LightGBM because it uses leaf-wise tree growth.

In [ ]:
def lgb_objective(trial):
    """Production LightGBM objective for Kaggle competitions."""
    params = {
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "random_state": 42,
        # Key LGB hyperparameters
        "num_leaves": trial.suggest_int("num_leaves", 20, 300),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "path_smooth": trial.suggest_float("path_smooth", 0.0, 1.0),
    }

    oof_scores = []
    for fold_idx, (train_idx, val_idx) in enumerate(CV.split(X, y)):
        X_fold_tr, X_fold_val = X[train_idx], X[val_idx]
        y_fold_tr, y_fold_val = y[train_idx], y[val_idx]

        train_set = lgb.Dataset(X_fold_tr, label=y_fold_tr)
        val_set = lgb.Dataset(X_fold_val, label=y_fold_val, reference=train_set)

        callbacks = [
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(-1),
        ]
        model = lgb.train(
            params,
            train_set,
            num_boost_round=params["n_estimators"],
            valid_sets=[val_set],
            callbacks=callbacks,
        )
        preds = model.predict(X_fold_val)
        oof_scores.append(roc_auc_score(y_fold_val, preds))

    return float(np.mean(oof_scores))


lgb_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="lgb_production",
)
lgb_study.optimize(lgb_objective, n_trials=40, show_progress_bar=True)

print(f"\nLightGBM best CV ROC-AUC: {lgb_study.best_value:.4f}")

In [ ]:
# Head-to-head comparison: XGBoost vs LightGBM
xgb_values = [t.value for t in xgb_study.trials if t.value is not None]
lgb_values = [t.value for t in lgb_study.trials if t.value is not None]

xgb_best_curve = pd.Series(xgb_values).cummax()
lgb_best_curve = pd.Series(lgb_values).cummax()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Convergence curves
axes[0].plot(xgb_best_curve.values, color=PALETTE[0], linewidth=2.5, label="XGBoost")
axes[0].plot(lgb_best_curve.values, color=PALETTE[1], linewidth=2.5, label="LightGBM")
axes[0].set_title("XGBoost vs LightGBM Convergence", fontweight="bold")
axes[0].set_xlabel("Trial Number")
axes[0].set_ylabel("Best CV ROC-AUC")
axes[0].legend()

# Boxplots of trial distributions
data_for_box = [
    pd.Series(xgb_values, name="XGBoost"),
    pd.Series(lgb_values, name="LightGBM"),
]
axes[1].boxplot(
    [xgb_values, lgb_values],
    labels=["XGBoost", "LightGBM"],
    patch_artist=True,
    boxprops=dict(facecolor=PALETTE[0], alpha=0.6),
    medianprops=dict(color="red", linewidth=2),
)
axes[1].set_title("Trial Value Distribution", fontweight="bold")
axes[1].set_ylabel("CV ROC-AUC")

plt.suptitle("XGBoost vs LightGBM: Optuna-Tuned Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"XGBoost  best: {xgb_study.best_value:.4f} (trial #{xgb_study.best_trial.number})")
print(f"LightGBM best: {lgb_study.best_value:.4f} (trial #{lgb_study.best_trial.number})")

<a id="10"></a>
## 10. Sklearn Integration: OptunaSearchCV

`OptunaSearchCV` is a drop-in replacement for scikit-learn's `RandomizedSearchCV` and `GridSearchCV`. It accepts standard sklearn estimators and exposes the familiar `.fit()`, `.best_params_`, and `.best_score_` interface.

**When to use it:**
- You are already using sklearn pipelines
- You want Optuna's sampling with sklearn's cross-validation machinery
- You need compatibility with sklearn's `Pipeline` and `ColumnTransformer`

### Distributions API

Instead of `trial.suggest_*`, `OptunaSearchCV` uses distribution objects:

```python
from optuna.distributions import (
    IntDistribution,       # was IntUniformDistribution in older Optuna
    FloatDistribution,     # was UniformDistribution / LogUniformDistribution
    CategoricalDistribution,
)
```

In [ ]:
from optuna.integration import OptunaSearchCV
from optuna.distributions import (
    IntDistribution,
    FloatDistribution,
    CategoricalDistribution,
)

param_distributions = {
    "n_estimators": IntDistribution(50, 400),
    "max_depth": IntDistribution(3, 12),
    "min_samples_split": IntDistribution(2, 30),
    "min_samples_leaf": IntDistribution(1, 15),
    "max_features": CategoricalDistribution(["sqrt", "log2"]),
    "min_impurity_decrease": FloatDistribution(0.0, 0.1),
}

opt_search = OptunaSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions,
    n_trials=30,
    cv=5,
    scoring="roc_auc",
    random_state=42,
    verbose=0,
    refit=True,  # refit best model on full dataset
)
opt_search.fit(X, y)

print(f"Best CV ROC-AUC : {opt_search.best_score_:.4f}")
print(f"\nBest parameters:")
for k, v in opt_search.best_params_.items():
    print(f"  {k}: {v}")

# The underlying study is accessible
inner_study = opt_search.study_
print(f"\nInner study: {len(inner_study.trials)} trials")

<a id="11"></a>
## 11. Multi-Objective Optimization

Real-world ML deployment involves **trade-offs**: you might want high accuracy *and* fast inference, or high AUC *and* low model complexity. Optuna supports multi-objective optimization natively.

### How It Works

- Specify multiple directions: `directions=["maximize", "minimize"]`
- Optuna finds the **Pareto front** -- the set of trials where no objective can be improved without sacrificing another
- `study.best_trials` returns all Pareto-optimal trials

### Practical Use Case: AUC vs Training Time

In Kaggle competitions with 9-hour kernels, you may want to find the sweet spot between model quality and compute budget.

In [ ]:
def multi_objective(trial):
    """Optimize AUC (maximize) and training time (minimize) simultaneously."""
    n_estimators = trial.suggest_int("n_estimators", 10, 300)
    max_depth = trial.suggest_int("max_depth", 2, 12)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)

    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
        n_jobs=-1,
    )
    t_start = time.time()
    auc = cross_val_score(clf, X, y, cv=3, scoring="roc_auc", n_jobs=-1).mean()
    elapsed = time.time() - t_start

    # Return (AUC, negative_time) -- both maximized
    return auc, -elapsed


mo_study = optuna.create_study(
    directions=["maximize", "maximize"],
    study_name="multi_objective",
    sampler=optuna.samplers.NSGAIISampler(seed=42),
)
mo_study.optimize(multi_objective, n_trials=50, show_progress_bar=True)

pareto_trials = mo_study.best_trials
print(f"Total trials  : {len(mo_study.trials)}")
print(f"Pareto front  : {len(pareto_trials)} trials")
print(f"\nSample Pareto-optimal solutions:")
for t in sorted(pareto_trials, key=lambda t: t.values[0], reverse=True)[:5]:
    auc_v = t.values[0]
    time_v = -t.values[1]
    print(f"  AUC={auc_v:.4f}, time={time_v:.2f}s | {t.params}")

In [ ]:
# Visualize the Pareto front
all_aucs = [t.values[0] for t in mo_study.trials if t.values is not None]
all_times = [-t.values[1] for t in mo_study.trials if t.values is not None]
pareto_aucs = [t.values[0] for t in pareto_trials]
pareto_times = [-t.values[1] for t in pareto_trials]

fig, ax = plt.subplots(figsize=(10, 6))

# All trials
ax.scatter(
    all_times, all_aucs,
    color="gray", alpha=0.35, s=25, label="All trials", zorder=2
)

# Pareto front
pareto_sorted = sorted(zip(pareto_times, pareto_aucs))
pt_times, pt_aucs = zip(*pareto_sorted)
ax.scatter(
    pt_times, pt_aucs,
    color=PALETTE[1], s=80, zorder=4, label="Pareto front", edgecolors="black"
)
ax.step(
    pt_times, pt_aucs,
    color=PALETTE[1], linewidth=2, where="post", zorder=3
)

ax.set_xlabel("Training Time (seconds)", fontsize=12)
ax.set_ylabel("CV ROC-AUC", fontsize=12)
ax.set_title(
    "Multi-Objective Optimization: AUC vs Training Speed\n"
    "(Pareto front = optimal trade-offs)",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

<a id="12"></a>
## 12. Hyperparameter Importance

Not all hyperparameters matter equally. Optuna's `get_param_importances` uses **fANOVA** (functional Analysis of Variance) to measure how much variance in the objective is explained by each hyperparameter.

This is extremely valuable for:
1. **Focusing future tuning** on the parameters that actually matter
2. **Reducing search space** by fixing unimportant parameters to reasonable defaults
3. **Understanding your model** -- which settings drive performance?

Rule of thumb: if a parameter explains < 1% of variance, fix it and save trials.

In [ ]:
# fANOVA importance from the TPE study
importances = optuna.importance.get_param_importances(tpe_study)

print("Hyperparameter Importances (fANOVA):")
print("-" * 55)
for param, imp in sorted(importances.items(), key=lambda x: -x[1]):
    bar = "█" * int(imp * 40)
    print(f"  {param:30s}: {bar:<40s} {imp:.3f}")
print("-" * 55)

# Visualize with bar chart
sorted_params = sorted(importances.items(), key=lambda x: x[1])
param_names = [p for p, _ in sorted_params]
imp_values = [v for _, v in sorted_params]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    param_names, imp_values,
    color=[PALETTE[0] if v > 0.05 else "lightgray" for v in imp_values],
    edgecolor="white"
)
ax.axvline(0.05, color="red", linestyle="--", linewidth=1.5,
           label="5% importance threshold")
ax.set_title("Hyperparameter Importances (fANOVA)", fontweight="bold", fontsize=13)
ax.set_xlabel("Relative Importance")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Interactive version (Plotly)
fig = optuna.visualization.plot_param_importances(tpe_study)
fig.update_layout(
    title="Hyperparameter Importance (Interactive)",
    height=400
)
fig.show()

# LightGBM importance for comparison
lgb_importances = optuna.importance.get_param_importances(lgb_study)
print("\nLightGBM Hyperparameter Importances:")
print("-" * 55)
for param, imp in sorted(lgb_importances.items(), key=lambda x: -x[1]):
    bar = "█" * int(imp * 40)
    print(f"  {param:30s}: {bar:<40s} {imp:.3f}")

<a id="13"></a>
## 13. Production Tips for Kaggle

### Best Practices at a Glance

| Tip | Why It Matters |
|-----|----------------|
| `n_jobs=-1` inside objective | Parallel CV; uses all Kaggle kernel CPUs |
| `optuna.logging.set_verbosity(WARNING)` | Clean output, no spam |
| `study.enqueue_trial(params)` | Seed with known-good params from literature |
| Use `timeout` alongside `n_trials` | Respects the 9-hour kernel limit |
| Save with `RDBStorage` (SQLite) | Resume study on kernel restart |
| `log=True` for learning rate | Log scale = equal density per order of magnitude |
| Set `n_startup_trials` >= 5 in pruner | Avoid pruning during cold start |
| Use `suggest_float(..., step=0.05)` | Discretize continuous params when needed |
| Run multiple parallel kernels | Use `storage=` to share the same study |
| Check `study.trials_dataframe()` | Find clusters of good configurations |

### Advanced: Enqueue Known-Good Trials

If you know from experience or prior work that certain hyperparameter values are good starting points, force Optuna to evaluate them first:

```python
study.enqueue_trial({
    "learning_rate": 0.05,
    "max_depth": 6,
    "n_estimators": 300,
    "subsample": 0.8,
})
study.optimize(objective, n_trials=100)
```

### Advanced: Persist Study Across Sessions

```python
import optuna

storage = optuna.storages.RDBStorage(
    url="sqlite:///optuna_study.db",
    engine_kwargs={"connect_args": {"timeout": 10}}
)

study = optuna.create_study(
    storage=storage,
    study_name="my_competition_study",
    direction="maximize",
    load_if_exists=True,  # Resume if study already exists
)
```

### Advanced: Distributed Optimization

Run multiple Kaggle kernels sharing the same SQLite/PostgreSQL study:

```python
# Kernel 1 and Kernel 2 both run this code
study.optimize(
    objective,
    n_trials=100,
    timeout=3600 * 8,   # 8-hour Kaggle kernel limit
    n_jobs=1,           # Let storage handle parallelism
)
```

Optuna handles concurrency automatically -- each kernel will pick up where others left off without duplicating work.

In [ ]:
# Demonstrate enqueue_trial: warm-start with a known-good configuration

warm_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="rf_warmstart",
)

# Enqueue a known-good configuration (e.g., from published paper or prior run)
warm_study.enqueue_trial({
    "n_estimators": 200,
    "max_depth": 8,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
})

warm_study.optimize(objective, n_trials=40, show_progress_bar=False)

print(f"Warm-start study best: {warm_study.best_value:.4f}")

# Trial 0 should be our enqueued configuration
trial_0 = warm_study.trials[0]
print(f"\nTrial 0 (enqueued): score={trial_0.value:.4f}, params={trial_0.params}")

# Compare warm vs cold start convergence
warm_vals = pd.Series([t.value for t in warm_study.trials]).cummax()
cold_vals = pd.Series([t.value for t in tpe_study.trials]).cummax()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(cold_vals.values[:40], color=PALETTE[0], linewidth=2.5, label="Cold start (TPE)")
ax.plot(warm_vals.values[:40], color=PALETTE[1], linewidth=2.5, label="Warm start (enqueue)")
ax.set_title("Cold Start vs Warm Start Convergence", fontweight="bold", fontsize=13)
ax.set_xlabel("Trial Number")
ax.set_ylabel("Best ROC-AUC")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate timeout: run for at most 30 seconds
timeout_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

t_start = time.time()
timeout_study.optimize(
    objective,
    n_trials=10000,    # large upper bound
    timeout=30,        # but stop after 30 seconds
    show_progress_bar=False,
)
elapsed = time.time() - t_start

print(f"Completed {len(timeout_study.trials)} trials in {elapsed:.1f}s (timeout=30s)")
print(f"Best value: {timeout_study.best_value:.4f}")
print(f"\nIn a 9-hour Kaggle kernel, set timeout=32400 (= 9 * 3600)")

---

## Key Takeaways

| Topic | What You Learned |
|-------|------------------|
| **TPE Sampler** | Bayesian optimization via density ratio; best general-purpose choice |
| **CMA-ES** | Evolution strategy for high-dim continuous spaces |
| **Pruning** | Kill bad trials early; 30-70% speedup with MedianPruner |
| **Visualization** | Interactive Plotly plots; matplotlib fallbacks |
| **XGBoost / LGB** | Full production objectives with StratifiedKFold + early stopping |
| **OptunaSearchCV** | Drop-in sklearn API; compatible with Pipelines |
| **Multi-Objective** | Pareto front; trade-off AUC vs speed/complexity |
| **Importance** | fANOVA tells you which params matter -- focus your budget there |
| **Production Tips** | Enqueue seeds, timeout, RDBStorage, distributed kernels |

### Recommended Optuna Workflow for Kaggle

```
1. Quick scan: 30 trials, wide bounds, RandomSampler  -> understand the space
2. Focused run: 100 trials, TPE, narrowed bounds      -> exploit promising regions
3. Importance check: fANOVA -> fix low-importance params
4. Final run: 200 trials, TPE, tight bounds, pruning  -> squeeze out final gains
5. Retrain best params on full train set              -> submit
```

### Further Reading

- [Optuna Official Documentation](https://optuna.readthedocs.io/)
- [Bergstra & Bengio (2012): Random Search for Hyper-Parameter Optimization](https://jmlr.org/papers/v13/bergstra12a.html)
- [Bergstra et al. (2011): Algorithms for Hyper-Parameter Optimization (TPE)](https://proceedings.neurips.cc/paper/2011/hash/86e8f7ab32cfd12577bc2619bc635690-Abstract.html)
- [Hansen (2016): The CMA Evolution Strategy](https://arxiv.org/abs/1604.00772)
- [Hutter et al. (2014): An Efficient Approach for Assessing Hyperparameter Importance (fANOVA)](http://proceedings.mlr.press/v32/hutter14.html)

---

### Portfolio Quality Addendum

This notebook is part of a systematic Kaggle Grandmaster portfolio covering:

- Feature Engineering Cookbook (50 techniques)
- Ensemble & Stacking Guide
- LLM Fine-tuning Cookbook (LoRA / QLoRA)
- Attention Mechanisms Walkthrough
- RAG from Scratch
- Graph Neural Networks
- **Optuna Hyperparameter Optimization Guide (this notebook)**

---

**If this notebook helped you, please upvote!** Feedback and comments are very welcome.

*Lorenzo Scaturchio | February 2026*